# 022-adaptive-gate

# Imports

In [ ]:
import json
import os

os.environ.setdefault("PYTORCH_ENABLE_MPS_FALLBACK", "1")

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from lopo import choose_p_max, fit_gate_on
from uavloc.data.drone_frames import DroneFrame, load_frames
from uavloc.data.multidate import load_date
from uavloc.datasets import dataset_ref
from uavloc.features.factory import correlator_config
from uavloc.features.segmentation import SegmentationFeatures, load_segmenter
from uavloc.matching.benchmark import benchmark_draws, score_draws, summarise
from uavloc.matching.correlator import PositionCorrelator
from uavloc.matching.risk import score_risk
from uavloc.matching.tuning import TuningGrid, tuned_config
from uavloc.track import REPO_ROOT, experiment_dir, log_run, set_seed

# Params

In [ ]:
EXP = "022-adaptive-gate"
NOTE = ""
LOG_RUN = True

ARM = "risk"                      # "static" | "risk"
GATE = "full"                     # static arm only: "full" (022.1) | "accept-reject" (022.2)
GATE_TAUS = [0.25, 0.5, 1.0, 2.0] # risk arm only: [0.25] (022.3) | the full list (022.4)
TEMPERATURE_GRID = [0.25, 0.5, 1.0, 2.0]   # static arm: leave-one-place-out grid, as 007
S_MAX_GRID_M = [2.0, 3.0, 5.0]
C_MIN_GRID = [0.01, 0.03, 0.05]
TUNING_NOISE_M = 10.0
STATIC_MIN_APPLIED_SHARE = 0.5
C_MIN = 0.01                      # risk arm: below this drone coverage the prior is kept
RIDGE_C = 1.0
P_MAX_GRID = [0.05, 0.10, 0.20, 0.30]
CV_FOLDS = 5
HARM_MARGIN_M = 2.0
MAX_HARM_SHARE = 0.042
MIN_APPLIED_SHARE = 0.60

PLACES = ["bushcha-50p2422-25p8305", "birky-49p9105-23p9192"]
MAP_DATE = "newest"
MATCHING = "weighted-ncc"
CLASS_WEIGHTS = (0.0, 1.0, 1.0, 0.5, 0.0, 0.5)   # background, building, track, woody, field, water
NOISE_M = [5.0, 10.0, 15.0]
PRIMARY_NOISE_M = 10.0
N_DIRECTIONS = 8
SEED = 42
DEVICE = "mps"
SEARCH_SCALE = 1.7
FOV_W_DEG = 60.0
SAT_FEATURE_CROP_PCT = 0.03
DRONE_FEATURE_CROP_PCT = 0.0
SEG_ARCHITECTURE = "lraspp"
SEG_CHECKPOINT = "pocs/mobilenet-lraspp-speed/model-lraspp-bn-satellite.pt"
SEG_CHANNELS = "all"
MATCH_GSD_M = 0.385
FRAMES_DATASET = "village-crossdate-frames"

# Data

In [ ]:
set_seed(SEED)

In [ ]:
RUN_CONFIG = {
    "method": "segmentation", "matching": MATCHING, "class_weights": CLASS_WEIGHTS,
    "noise_m": NOISE_M, "primary_noise_m": PRIMARY_NOISE_M, "n_directions": N_DIRECTIONS,
    "window": "rect", "search_scale": SEARCH_SCALE, "search_radius_m": None, "prior_sigma_m": None,
    "arm": ARM, "gate": GATE if ARM == "static" else "risk", "gate_taus": GATE_TAUS if ARM == "risk" else None,
    "tuning": {"temperatures": TEMPERATURE_GRID, "s_max_m": S_MAX_GRID_M, "c_min": C_MIN_GRID, "noise_m": TUNING_NOISE_M,
               "min_applied_share": STATIC_MIN_APPLIED_SHARE, "scheme": "leave-one-place-out", "temperature_by": "mean-error"},
    "risk_c_min": C_MIN,
    "fov_w_deg": FOV_W_DEG, "sat_feature_crop_pct": SAT_FEATURE_CROP_PCT, "drone_feature_crop_pct": DRONE_FEATURE_CROP_PCT,
    "seg_architecture": SEG_ARCHITECTURE, "seg_checkpoint_sat": SEG_CHECKPOINT, "seg_checkpoint_drone": SEG_CHECKPOINT,
    "seg_channels": SEG_CHANNELS, "drone_seg_gsd_m": MATCH_GSD_M, "match_gsd_m": MATCH_GSD_M,
    "frames_dataset": FRAMES_DATASET, "frames_dataset_ref": dataset_ref(FRAMES_DATASET),
    "places": PLACES, "map_date": MAP_DATE,
    "risk": {"ridge_c": RIDGE_C, "p_max_grid": P_MAX_GRID, "cv_folds": CV_FOLDS, "harm_margin_m": HARM_MARGIN_M,
             "max_harm_share": MAX_HARM_SHARE, "min_applied_share": MIN_APPLIED_SHARE, "scheme": "leave-one-place-out"},
}

In [ ]:
def frame_place(frame: DroneFrame) -> str:
    return json.loads((frame.image_path.parent / "meta.json").read_text())["place"]

In [ ]:
frames = [f for f in load_frames(FRAMES_DATASET) if frame_place(f) in PLACES]
places = {place: [f for f in frames if frame_place(f) == place] for place in PLACES}
maps = {place: load_date(place, MAP_DATE) for place in PLACES}
{place: len(members) for place, members in places.items()}

# Model

In [ ]:
model = load_segmenter(REPO_ROOT / SEG_CHECKPOINT, DEVICE, SEG_ARCHITECTURE)

In [ ]:
features = {place: SegmentationFeatures(model, model, maps[place], SEG_CHANNELS, MATCH_GSD_M, MATCH_GSD_M, DEVICE)
            for place in PLACES}
base_config = correlator_config({**RUN_CONFIG, "gate": "none"})
correlators = {place: PositionCorrelator(features[place], base_config) for place in PLACES}
base_config

# Run

In [ ]:
draws = {place: [d for noise_m in NOISE_M
                 for d in benchmark_draws(correlators[place], places[place], noise_m, N_DIRECTIONS, SEED)]
         for place in PLACES}
{place: len(d) for place, d in draws.items()}

In [ ]:
def other_place(held_out: str) -> str:
    """Leave-one-place-out: a held-out place is scored with values fitted on the other one."""
    return next(place for place in PLACES if place != held_out)

In [ ]:
grid = TuningGrid(tuple(TEMPERATURE_GRID), tuple(S_MAX_GRID_M), tuple(C_MIN_GRID), TUNING_NOISE_M,
                  STATIC_MIN_APPLIED_SHARE)

In [ ]:
static_folds = {place: tuned_config(draws[other_place(place)], base_config, GATE, grid)
                for place in PLACES} if ARM == "static" else {}
{place: (c.temperature, c.gate_s_max_m, c.gate_c_min) for place, (c, _) in static_folds.items()}

In [ ]:
def risk_fold(held_out: str):
    """Risk model and `p_max` for one held-out place, from the other place's draws only."""
    train = draws[other_place(held_out)]
    p_max, tuning = choose_p_max(train, base_config, tuple(GATE_TAUS), RIDGE_C, P_MAX_GRID, CV_FOLDS, HARM_MARGIN_M,
                                 PRIMARY_NOISE_M, MAX_HARM_SHARE, MIN_APPLIED_SHARE, C_MIN)
    return fit_gate_on(train, base_config, tuple(GATE_TAUS), RIDGE_C, HARM_MARGIN_M, C_MIN), p_max, tuning

In [ ]:
folds = {place: risk_fold(place) for place in PLACES} if ARM == "risk" else {}
{place: p_max for place, (_, p_max, _) in folds.items()}

In [ ]:
def score_place(place: str) -> pd.DataFrame:
    if ARM == "static":
        return score_draws(draws[place], static_folds[place][0])
    gate, p_max, _ = folds[place]
    return score_risk(draws[place], base_config, gate, p_max, tuple(GATE_TAUS), C_MIN)

In [ ]:
records = pd.concat([score_place(place).assign(place=place) for place in PLACES], ignore_index=True)
records["kept_prior"] = records["decision"] == "reject"
records["harm"] = records["error_m"] > records["prior_error_m"] + HARM_MARGIN_M
records.groupby(["place", "noise_m"])[["error_m", "harm", "kept_prior", "reachable"]].mean().round(3)

In [ ]:
pd.crosstab([records["noise_m"], records["place"]], records["decision"], normalize="index").round(3)

In [ ]:
def run_metrics(records: pd.DataFrame) -> dict[str, float]:
    """005/006's metrics (harm and decision shares included), plus the share of draws that kept the prior."""
    subsets = {"all": {f.name for f in frames}, **{p.split("-")[0]: {f.name for f in m} for p, m in places.items()}}
    metrics = summarise(records, subsets, PRIMARY_NOISE_M)
    for noise_m, group in records.groupby("noise_m"):
        metrics[f"kept_prior_share_at_{noise_m:g}m"] = float(group["kept_prior"].mean())
    metrics["kept_prior_share"] = metrics[f"kept_prior_share_at_{PRIMARY_NOISE_M:g}m"]
    return metrics

In [ ]:
metrics = run_metrics(records)
{k: round(v, 3) for k, v in metrics.items() if "_at_" not in k}

# Plots

In [ ]:
fig_cdf, ax = plt.subplots(figsize=(6, 4))
for noise_m, group in records.groupby("noise_m"):
    errors = np.sort(group["error_m"].to_numpy())
    line, = ax.plot(errors, np.linspace(0, 1, len(errors)), label=f"{noise_m:g} m injected")
    ax.axvline(noise_m, color=line.get_color(), ls=":", lw=1)
ax.set_xlabel("position error after correction, m"); ax.set_ylabel("share of draws")
ax.set_title(f"arm {ARM}, gate {RUN_CONFIG['gate']}, taus {RUN_CONFIG['gate_taus']}"); ax.legend(); ax.grid(alpha=.3)
fig_cdf.tight_layout()

# Log

In [ ]:
artifacts = experiment_dir(EXP) / "artifacts"
artifacts.mkdir(exist_ok=True)
RUN_CONFIG["p_max"] = {place: p_max for place, (_, p_max, _) in folds.items()} or None
RUN_CONFIG["risk_weights"] = {place: gate.weights() for place, (gate, _, _) in folds.items()} or None
for field in ("temperature", "gate_s_max_m", "gate_c_min"):
    RUN_CONFIG[field] = {place: getattr(c, field) for place, (c, _) in static_folds.items()} or None
run = log_run(exp=EXP, config=RUN_CONFIG, metrics=metrics, note=NOTE, figures=[fig_cdf],
              seed=SEED, device=DEVICE) if LOG_RUN else {"run": "dry"}
records.to_csv(artifacts / f"records-{run['run']}.csv", index=False)
run["run"]

In [ ]:
tuning_tables = [t.assign(held_out=place) for place, (_, _, t) in folds.items()] + \
                 [t.assign(held_out=place) for place, (_, t) in static_folds.items()]
pd.concat(tuning_tables, ignore_index=True).to_csv(artifacts / f"tuning-{run['run']}.csv", index=False)